# Prova pilot GPT-5 RGB JSON

Aquest notebook prova la familia `gpt-5` i incorpora els resultats ja existents de `gpt-4o` i `gpt-4o-mini` amb un prompt que demana RGB numeric separat en JSON. La idea es comprovar si baixa la quantitzacio de respostes repetides com `00FF00`, `0000FF` o `FF00FF`.

No genera colors ni imatges noves: reutilitza `recollida-dades/experiments/soroll/csv/input_image_sample.csv` i `recollida-dades/experiments/soroll/images/`. Les sortides queden separades dins `recollida-dades/experiments/soroll/model-runs/gpt5-family-rgb-json/` per reutilitzar els resultats ja generats.

## Configuracio

Per defecte `RUN_MODEL_QUERIES` esta a `False` per evitar crides accidentals. Canvia'l a `True` quan vulguis executar la prova pilot.

In [ ]:
from pathlib import Path
import sys
import time

RUN_MODEL_QUERIES = True
GPT5_RGB_MODELS = ["gpt-5", "gpt-5.1", "gpt-5.2", "gpt-5.4", "gpt-5.5"]
GPT5_RGB_TEMPERATURE = None  # GPT-5 no usa temperature; no enviem aquest parametre.
GPT5_RGB_REASONING_EFFORT = "low"  # nomes s'envia als models gpt-5*; evita el valor minimal.
GPT5_RGB_PILOT_IMAGES = 1000
GPT5_RGB_PILOT_SEED = 23
BASELINE_MODELS = ["gpt-4o", "gpt-4o-mini"]
GPT5_RGB_MAX_OUTPUT_TOKENS = 1200  # evita JSON tallat en models raonadors


def _format_duration_local(seconds):
    seconds = max(0, float(seconds))
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes, remaining_seconds = divmod(seconds, 60)
    if minutes < 60:
        return f"{int(minutes)}m {remaining_seconds:04.1f}s"
    hours, remaining_minutes = divmod(minutes, 60)
    return f"{int(hours)}h {int(remaining_minutes)}m {remaining_seconds:04.1f}s"


def _install_cell_timer():
    ipython = get_ipython()
    if ipython is None or getattr(ipython, "_color_llm_timer_installed", False):
        return

    def _start_cell_timer(info):
        ipython.user_ns["_cell_started_at"] = time.perf_counter()

    def _stop_cell_timer(result):
        started_at = ipython.user_ns.get("_cell_started_at")
        if started_at is not None:
            elapsed = time.perf_counter() - started_at
            print(f"Temps cel.la: {_format_duration_local(elapsed)}")

    ipython.events.register("pre_run_cell", _start_cell_timer)
    ipython.events.register("post_run_cell", _stop_cell_timer)
    ipython._color_llm_timer_installed = True


_install_cell_timer()

base_dir = Path("recollida-dades")
if not base_dir.exists():
    base_dir = Path("..").resolve()
if not (base_dir / "scripts").exists():
    raise FileNotFoundError(f"No trobo recollida-dades/scripts des de {Path.cwd()}")
source_csv_dir = base_dir / "experiments" / "soroll" / "csv"
source_images_dir = base_dir / "experiments" / "soroll" / "images"
pilot_dir = base_dir / "experiments" / "soroll" / "model-runs" / "gpt5-family-rgb-json"
csv_dir = pilot_dir / "csv"
logs_dir = pilot_dir / "logs"
tmp_dir = pilot_dir / "tmp"
scripts_dir = base_dir / "scripts"

for folder in [csv_dir, logs_dir, tmp_dir]:
    folder.mkdir(parents=True, exist_ok=True)

if str(scripts_dir) not in sys.path:
    sys.path.append(str(scripts_dir))

print("Carpeta base:", base_dir)
print("Carpeta pilot:", pilot_dir)


## Carrega de scripts i dades

Carreguem funcions comunes, la mostra original i, si existeix, el resultat antic per seleccionar casos dificils.

In [ ]:
import importlib
import pandas as pd
from IPython.display import Image as DisplayImage, display

import utils
importlib.reload(utils)

build_final_sample = utils.build_final_sample
collect_model_outputs = utils.collect_model_outputs
save_csv = utils.save_csv
save_error_distribution = utils.save_error_distribution
save_error_boxplot = utils.save_error_boxplot
save_response_repetition_plot = utils.save_response_repetition_plot
save_error_vs_chroma_plot = utils.save_error_vs_chroma_plot
select_pilot_image_paths = utils.select_pilot_image_paths
write_log = utils.write_log

required_functions = [
    "build_final_sample",
    "collect_model_outputs",
    "save_csv",
    "save_error_distribution",
    "save_error_boxplot",
    "save_response_repetition_plot",
    "save_error_vs_chroma_plot",
    "select_pilot_image_paths",
    "write_log",
]
missing = [name for name in required_functions if name not in globals()]
if missing:
    raise RuntimeError(f"Error: scripts no carregats correctament: {missing}")

input_path = source_csv_dir / "input_image_sample.csv"
previous_results_path = source_csv_dir / "sample-colors_4o.csv"

if not input_path.exists():
    raise FileNotFoundError(f"No trobo la mostra original: {input_path}")
if not source_images_dir.exists():
    raise FileNotFoundError(f"No trobo les imatges originals: {source_images_dir}")

input_sample = pd.read_csv(input_path)
previous_results = pd.read_csv(previous_results_path) if previous_results_path.exists() else None

print(f"Mostra original: {len(input_sample)} colors")
print(f"Resultats previs disponibles: {previous_results is not None}")
write_log("Notebook GPT-5 RGB pilot inicialitzat", logs_dir / "pipeline.log")


## Seleccio de la mostra pilot

La mostra pilot combina casos dificils de l'experiment anterior amb colors repartits per `chroma`, per no provar nomes els primers fitxers alfabeticament.

In [ ]:
pilot_image_paths = select_pilot_image_paths(
    sample=input_sample,
    images_dir=source_images_dir,
    n_images=GPT5_RGB_PILOT_IMAGES,
    seed=GPT5_RGB_PILOT_SEED,
    previous_results=previous_results,
)

pilot_image_names = [path.name for path in pilot_image_paths]
pilot_input_sample = input_sample[input_sample["image_name"].isin(pilot_image_names)].copy()

print(f"Imatges pilot seleccionades: {len(pilot_image_paths)}")
print(pilot_image_names[:15])

if len(pilot_image_paths) != GPT5_RGB_PILOT_IMAGES:
    raise RuntimeError("La mostra pilot no te el nombre d'imatges esperat.")

display(pilot_input_sample[["image_name", "hex", "r", "g", "b", "chroma"]].head())


## Consulta als models GPT-5

Executa nomes quan `RUN_MODEL_QUERIES = True`. Guarda resultats incrementals per poder reprendre si s'atura a mig cami.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

output_path = csv_dir / "outputmodel_image_sample_gpt5_family_rgb_json.csv"
final_path = csv_dir / "sample-colors-gpt5-family-rgb-json.csv"
model_log_path = logs_dir / "model_calls_gpt5_family_rgb_json.log"

if RUN_MODEL_QUERIES:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Falta OPENAI_API_KEY. Crea un fitxer .env local amb la clau abans d'executar models.")

    client = OpenAI()
    model_outputs = collect_model_outputs(
        client=client,
        image_paths=pilot_image_paths,
        models=GPT5_RGB_MODELS,
        temperature=GPT5_RGB_TEMPERATURE,
        output_path=output_path,
        log_file=model_log_path,
        response_format="rgb_json",
        retry_failed=True,
        reasoning_effort=GPT5_RGB_REASONING_EFFORT,
        max_output_tokens=GPT5_RGB_MAX_OUTPUT_TOKENS,
    )

    print(f"Resultats guardats: {output_path}")
    print(f"Files totals: {len(model_outputs)}")
    display(model_outputs.tail())
else:
    if output_path.exists():
        model_outputs = pd.read_csv(output_path)
        print(f"Models no executats ara. Carrego resultats existents: {len(model_outputs)} files")
        display(model_outputs.tail())
    else:
        model_outputs = pd.DataFrame()
        print("Models no executats. Posa RUN_MODEL_QUERIES = True quan vulguis cridar l'API.")


## Dataset final del pilot

Uneix la mostra pilot amb les respostes i calcula l'error cromatic. Tambe mostra metriques rapides de quantitzacio.

In [ ]:
final_path = csv_dir / "sample-colors-gpt5-family-rgb-json.csv"

if output_path.exists():
    model_outputs = pd.read_csv(output_path)
    final_sample = build_final_sample(pilot_input_sample, model_outputs, images_dir=source_images_dir)

    baseline_sample = pd.DataFrame()
    if previous_results is not None:
        baseline_sample = previous_results[
            previous_results["model"].isin(BASELINE_MODELS)
            & previous_results["image_name"].isin(pilot_input_sample["image_name"])
        ].copy()
    if not baseline_sample.empty:
        final_sample = pd.concat([baseline_sample, final_sample], ignore_index=True, sort=False)
        model_order = BASELINE_MODELS + GPT5_RGB_MODELS
        final_sample["model"] = pd.Categorical(final_sample["model"], categories=model_order, ordered=True)
        final_sample = final_sample.sort_values(["image_name", "model"]).reset_index(drop=True)
        final_sample["model"] = final_sample["model"].astype(str)
    save_csv(final_sample, final_path)
    write_log(f"Dataset final GPT-5 RGB pilot guardat: {final_path} files={len(final_sample)}", logs_dir / "pipeline.log")

    print(f"Dataset final guardat: {final_path}")

    status_summary = final_sample.groupby(["model", "status"], dropna=False).size().reset_index(name="files")
    unique_summary = final_sample.groupby("model")["response_hex"].nunique().reset_index(name="response_hex_unics")
    error_summary = final_sample.groupby("model")["error_cromatic"].agg(["count", "mean", "median", "min", "max"]).reset_index()
    repeated_summary = (
        final_sample.groupby("model")["response_hex"]
        .value_counts()
        .reset_index(name="repeticions")
        .query("repeticions > 1")
        .sort_values(["model", "repeticions"], ascending=[True, False])
    )

    display(status_summary)
    display(unique_summary)
    display(error_summary)
    display(repeated_summary.head(20))
    display(final_sample.head())

    error_distribution_path = tmp_dir / "error_distribution_gpt5_family_rgb.png"
    error_boxplot_path = tmp_dir / "error_boxplot_gpt5_family_rgb.png"
    response_repetition_path = tmp_dir / "response_repetition_gpt5_family_rgb.png"
    error_vs_chroma_path = tmp_dir / "error_vs_chroma_gpt5_family_rgb.png"

    save_error_distribution(final_sample, error_distribution_path)
    save_error_boxplot(final_sample, error_boxplot_path)
    save_response_repetition_plot(final_sample, response_repetition_path)
    save_error_vs_chroma_plot(final_sample, error_vs_chroma_path)

    print("Grafics generats:")
    print(error_distribution_path)
    print(error_boxplot_path)
    print(response_repetition_path)
    print(error_vs_chroma_path)

    display(DisplayImage(filename=str(error_distribution_path)))
    display(DisplayImage(filename=str(error_boxplot_path)))
    display(DisplayImage(filename=str(response_repetition_path)))
    display(DisplayImage(filename=str(error_vs_chroma_path)))
else:
    final_sample = pd.DataFrame()
    print("Encara no hi ha resultats del pilot. Executa la cel.la de models amb RUN_MODEL_QUERIES = True.")


## Fitxers generats

Comprovacio rapida del que ha quedat dins la carpeta separada del pilot.

In [ ]:
print("CSV pilot:", sorted(path.name for path in csv_dir.glob("*.csv")))
print("Logs pilot:", sorted(path.name for path in logs_dir.glob("*.log")))
print("Tmp pilot:", sorted(path.name for path in tmp_dir.glob("*")))
print("Grafics PNG:", sorted(path.name for path in tmp_dir.glob("*.png")))
